# Imports

In [104]:
import numpy as np
import pandas as pd
import keras
import torch
import tensorflow as tf
import transformers
import sentence_transformers as st
import tqdm.notebook as tqdm

# Load data using pandas

In [105]:
query = pd.read_parquet('query.parquet.brotli')["query"].tolist()
answer = pd.read_parquet('answer.parquet.brotli')["shuffled_answer"].tolist()

In [106]:
'''print(query.head())
print(answer.head())
print(query.shape, answer.shape)'''

'print(query.head())\nprint(answer.head())\nprint(query.shape, answer.shape)'

In [107]:
#model = st.SentenceTransformer('all-MiniLM-L6-v2', device="cuda")
model = st.SentenceTransformer('BAAI/bge-large-en-v1.5', device="cuda")

In [108]:
query_embeddings = model.encode(
    query,
    #prompt="Given a query, retrieve relevant documents: ",
    #batch_size=1024
    batch_size=2048,
    show_progress_bar=True,
    device="cuda",
    normalize_embeddings=True,
)
np.savez('query_emb.npz', query_embeddings)

answer_embeddings = model.encode(
    answer,
    batch_size=128,
    show_progress_bar=True,
    device="cuda",
    normalize_embeddings=True,
)
np.savez('answer_emb.npz', answer_embeddings)

Batches:   0%|          | 0/25 [00:00<?, ?it/s]

Batches:   0%|          | 0/588 [00:00<?, ?it/s]

# Ranking

Apply some dark magic to select 5 candidates to answer each query.

In [109]:
tensor_query_embeddings = torch.from_numpy(query_embeddings).to("cuda")
tensor_answer_embeddings = torch.from_numpy(answer_embeddings).to("cuda")

In [110]:
#scores = (tensor_query_embeddings @ tensor_answer_embeddings.T).cpu()

In [111]:
QUERY_BATCH_SIZE = 32
top5_indices = []

for i in range(0, len(query_embeddings), QUERY_BATCH_SIZE):
    batch_queries = torch.from_numpy(query_embeddings[i:i + QUERY_BATCH_SIZE]).to("cuda")

    batch_scores = batch_queries @ tensor_answer_embeddings.T

    _, i = torch.topk(batch_scores, k=5, dim=-1)
    top5_indices.append(i.cpu().numpy())

    # Очистка памяти
    del batch_queries, batch_scores, i
    torch.cuda.empty_cache()

In [112]:
#_, i = torch.topk(scores, k=5, axis=-1)
#print(i.shape)

In [113]:
top5_indices = np.vstack(top5_indices)

In [114]:
#np.savez('pred.npz', i)

In [115]:
np.savez('final_pred.npz', top5_indices)

In [116]:
pred_ = np.load('final_pred.npz')['arr_0']
i = 0
print("Вопрос:", query[i])
print("Топ-5 ответов:")
for j, ans_idx in enumerate(pred_[i]):
    print(f"{j+1}. {answer[ans_idx]}")

Вопрос: when did beavis and butthead first come out
Топ-5 ответов:
1. Beavis and Butt-Head Beavis and Butt-Head is an American animated sitcom created and designed by Mike Judge.[1] The series originated from Frog Baseball, a 1992 short film by Judge originally aired on Liquid Television. After seeing the short, MTV signed Judge to develop the concept.[2][3] The series first ran from March 8, 1993, to November 28, 1997. The series was later renewed for an eighth season, which aired from October 27 to December 29, 2011. In 1996, the series was adapted into the animated feature film Beavis and Butt-Head Do America.
2. Help Wanted (SpongeBob SquarePants) "Help Wanted" is the pilot episode of the American animated television series SpongeBob SquarePants. It originally aired on Nickelodeon in the United States on May 1, 1999, following the television airing of the 1999 Kids' Choice Awards. The episode follows the protagonist, an anthropomorphic sea sponge named SpongeBob SquarePants, attemp

In [117]:
pred_

array([[ 6868, 12632, 25199, 75062, 35796],
       [66994, 23401, 65909, 24850, 74747],
       [13702, 57806, 72024, 11046, 31043],
       ...,
       [55207,   352, 37920, 51928,  3369],
       [   39, 64951, 39088,  2069, 34398],
       [59706, 31211, 74344, 30054, 27114]])

# Save predictions

Save predictions file and upload to Cats

In [118]:
np.load('pred.npz')['arr_0'].shape

FileNotFoundError: [Errno 2] No such file or directory: 'pred.npz'